# CÍL 4: NumPy

> Tento notebook je součástí cvičení 12 předmětu BPC-PRG.  
> Projdi ho postupně – každou buňku spusť klávesou **Shift+Enter**.  
> 📥 **[Stáhnout notebook (cviceni_12_numpy.ipynb)](cviceni_12_numpy.ipynb)** &nbsp;|&nbsp; 📥 **[Stáhnout data (pacienti.npy)](../pacienti.npy)**
>
> Data potřebuješ pro závěrečný úkol — stáhni si `pacienti.npy` vedle notebooku.


## Co je NumPy?

[**NumPy**](https://numpy.org/) *(Numerical Python)* je open-source matematická knihovna jazyka Python, na které stojí celý vědecký Python ekosystém. Vznikla v roce 2005 a dnes ji využívají datové vědy, fyzika, bioinformatika i strojové učení.

Jejím jádrem je datová struktura **`ndarray`** (N-dimensional array) – kompaktní pole čísel uložené přímo v paměti jako v jazyku C. Na rozdíl od Python listu ukládá všechny prvky stejného datového typu bez doprovodných objektů a metadat a operace nad ním jsou implementovány v kompilovaném C/Fortran kódu.

> **Tip:** NumPy v Pythonu z velké části nahrazuje MATLAB – syntaxe je velmi podobná, ale Python je volně dostupný a aktivně vyvíjený.

### Kde NumPy potkáš v biomedicíně?

| Oblast                                  | Popis                                                     |
|-----------------------------------------|-----------------------------------------------------------|
| Zpracování signálů (EKG, EEG, SpO₂)     | Časové řady jako 1D pole, FFT, filtrování                 |
| Medicínské snímky (MRI, CT, histologie) | 2D/3D matice pixelů, geometrické transformace             |
| Klinická data a biostatistika           | Rychlé výpočty průměrů, std, korelací na tisících záznamů |
| Strojové učení a AI                     | Všechny datové sady jsou interně numpy pole               |

### Místo v Python ekosystému

NumPy je základ, na kterém staví ostatní vědecké knihovny:

```
NumPy ──► Matplotlib   (grafy, vizualizace)
      ──► Pandas       (tabulková data, DataFrame)
      ──► SciPy        (vědecké algoritmy – FFT, filtry, statistická analýza)
      ──► scikit-learn (strojové učení)
      ──► TensorFlow / PyTorch  (hluboké neuronové sítě)
```

Standardní import – zkratka `np` je všeobecnou konvencí:

```python
import numpy as np
```

## 2.1 Proč numpy a ne Python seznam?

**numpy pole (`ndarray`)** je základní datová struktura knihovny NumPy. Na rozdíl od Python listu:

- uchovává data v **kompaktním bloku paměti stejného datového typu** (např. všechna `float64`),
- operace jsou implementované v C/Fortran – Python smyčka není potřeba,
- mohou být **vícerozměrná** (2D matice, 3D tenzory…),
- podporují přímou **matematickou syntaxi**: `a * 2`, `a + b`, `np.sin(a)` – bez `for` cyklu.

Výsledek: **10–100× rychlejší** pro numerické operace a výrazně čitelnější kód.

Srovnání syntaxe – násobení každého prvku dvěma:

```python
# Python list – musíš projít každý prvek
result = [x * 2 for x in my_list]

# NumPy – přímá matematická operace na celém poli
result = my_array * 2
```

In [1]:
import numpy as np
import time

n = 10_000_000
py_list  = list(range(n))
np_array = np.arange(n)

# Varianta 1 – klasická for smyčka
result_loop = [0] * n
start = time.time()
for i in range(n):
    result_loop[i] = py_list[i] * 2
t_loop = time.time() - start

# Varianta 2 – list comprehension
start = time.time()
result_lc = [x * 2 for x in py_list]
t_lc = time.time() - start

# Varianta 3 – NumPy vektorizace
start = time.time()
result_np = np_array * 2
t_np = time.time() - start

print(f"for smyčka:         {t_loop:.3f} s")
print(f"list comprehension: {t_lc:.3f} s")
print(f"NumPy:              {t_np:.3f} s")
print(f"\nNumPy vs for smyčka:        {t_loop / t_np:.0f}×")
print(f"NumPy vs list comprehension: {t_lc / t_np:.0f}×")


for smyčka:         1.420 s
list comprehension: 1.283 s
NumPy:              0.079 s

NumPy vs for smyčka:        18×
NumPy vs list comprehension: 16×


## 2.2 Vytvoření pole

In [6]:
import numpy as np

# Ze seznamu
a = np.array([1, 2, 3, 4, 5])
a


array([1, 2, 3, 4, 5])

In [5]:
# Číselná řada – arange(start, stop, krok)
np.arange(0, 10, 2)


array([0, 2, 4, 6, 8])

In [4]:
# Rovnoměrně rozložené hodnoty – ideální pro časové osy
np.linspace(0, 1, 6)   # 6 hodnot od 0 do 1 včetně


array([0. , 0.2, 0.4, 0.6, 0.8, 1. ])

In [3]:
np.zeros(4)


array([0., 0., 0., 0.])

In [2]:
np.ones((2, 3))


array([[1., 1., 1.],
       [1., 1., 1.]])

In [7]:
# Gaussův šum – realistické biomedicínské hodnoty
noise = np.random.normal(loc=0, scale=0.1, size=8)
np.round(noise, 3)


array([-0.029,  0.294, -0.121,  0.161, -0.102,  0.068, -0.062, -0.027])

## 2.3 Tvar, datové typy a reshape

### Tvar a indexování – 1D pole

In [8]:
signal = np.array([0.5, 1.2, 1.8, 0.9, 2.1, 1.5, 0.7, 1.1])
signal


array([0.5, 1.2, 1.8, 0.9, 2.1, 1.5, 0.7, 1.1])

In [9]:
signal.shape, signal.ndim, len(signal)


((8,), 1, 8)

In [10]:
signal[0], signal[-1]   # první a poslední prvek


(np.float64(0.5), np.float64(1.1))

In [11]:
signal[2:5]   # prvky od indexu 2 po index 4


array([1.8, 0.9, 2.1])

In [12]:
signal[::2]   # každý druhý prvek


array([0.5, 1.8, 2.1, 0.7])

### 2D pole (matice)

In [13]:
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])
matrix


array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [14]:
matrix.shape


(3, 3)

In [15]:
matrix[1, 2]   # druhý řádek, třetí sloupec → 6


np.int64(6)

In [16]:
matrix[0, :]   # první řádek


array([1, 2, 3])

In [17]:
matrix[:, 1]   # druhý sloupec


array([2, 5, 8])

### Datové typy (dtype)

In [18]:
a_int = np.array([1, 2, 3])
a_int.dtype


dtype('int64')

In [19]:
a_float = np.array([1.0, 2.0, 3.0])
a_float.dtype


dtype('float64')

In [20]:
# Explicitní nastavení – float32 ušetří paměť oproti float64
a32 = np.zeros(4, dtype=np.float32)
print(f"dtype: {a32.dtype}   paměť na prvek: {a32.itemsize} B")


dtype: float32   paměť na prvek: 4 B


In [21]:
# Konverze dtype pomocí astype
a_int.astype(float)


array([1., 2., 3.])

### Změna tvaru (reshape)

In [22]:
flat = np.arange(12)
flat


array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [23]:
flat.reshape(3, 4)   # 3 řádky × 4 sloupce


array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [24]:
flat.reshape(2, 2, 3).shape   # 3D tenzor


(2, 2, 3)

In [25]:
# -1 = NumPy dopočítá rozměr automaticky
flat.reshape(1, -1).shape, flat.reshape(-1, 1).shape


((1, 12), (12, 1))

## 2.4 Pokročilé indexování

Kromě klasického slicingu nabízí NumPy dva mocné způsoby výběru prvků:

- **Fancy indexing** – výběr pomocí pole celých čísel (libovolné pozice),
- **Boolean (binární) indexing** – výběr pomocí masky True/False (filtrování dle podmínky).

### Fancy indexing

In [26]:
data = np.array([10, 20, 30, 40, 50, 60, 70, 80])
data[[0, 2, 5]]   # prvky na pozicích 0, 2, 5


array([10, 30, 60])

In [27]:
data[[5, 0, 5, 2]]   # libovolné pořadí, opakování je OK


array([60, 10, 60, 30])

In [28]:
# Výběr konkrétních pacientů z matice měření
mereni = np.array([[120, 80, 72],   # pacient 0: systola, diastola, TF
                   [135, 88, 90],   # pacient 1
                   [118, 76, 65],   # pacient 2
                   [145, 95, 88]])  # pacient 3

mereni[[0, 2]]   # pacienti 0 a 2


array([[120,  80,  72],
       [118,  76,  65]])

### Boolean (binární) indexing

In [29]:
# Maska – pole True/False stejné délky jako data
mask = np.array([True, False, True, False, True, False, True, False])
data[mask]


array([10, 30, 50, 70])

In [30]:
# Masku nejčastěji vytvoříme podmínkou přímo
data > 40


array([False, False, False, False,  True,  True,  True,  True])

In [31]:
data[data > 40]


array([50, 60, 70, 80])

In [32]:
# Kombinace podmínek: & = AND,  | = OR
data[(data > 20) & (data < 70)]


array([30, 40, 50, 60])

## 2.5 Matematické operace

NumPy operace jsou **vektorizované** – pracují na celém poli naráz, bez explicitního cyklu.

In [ ]:
a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([0.5, 0.5, 0.5, 0.5])

a + b


In [ ]:
a ** 2


In [ ]:
np.round(np.sqrt(a), 3)


In [ ]:
print(f"sum: {a.sum()}   mean: {a.mean()}   std: {a.std():.3f}")
print(f"min: {a.min()}   max: {a.max()}")


In [ ]:
# Podmínkové indexování a agregace v jednom výrazu
a[a > 2].mean()


### Agregace přes osy (`axis`)

U 2D matic můžeš agregovat buď přes celé pole, nebo přes jednu osu pomocí parametru `axis`:

- `axis=0` – sčítá / průměruje **přes řádky** → výsledkem je hodnota pro každý sloupec,
- `axis=1` – sčítá / průměruje **přes sloupce** → výsledkem je hodnota pro každý řádek.

In [ ]:
mereni = np.array([[120, 80, 72],   # pacient 0: systola, diastola, TF
                   [135, 88, 90],   # pacient 1
                   [118, 76, 65],   # pacient 2
                   [145, 95, 88]])  # pacient 3

print("Průměr přes všechna data:        ", mereni.mean())
print("Průměr po sloupcích (axis=0):    ", mereni.mean(axis=0))   # průměr každého ukazatele
print("Průměr po řádcích (axis=1):      ", mereni.mean(axis=1))   # průměr každého pacienta


### `argmin` / `argmax` – kde je minimum a maximum

Funkce `np.argmin` a `np.argmax` nevrací samotnou hodnotu, ale **index**, na kterém leží.
Hodí se vždy, když potřebuješ vědět *„kde"* (např. čas R-vrcholu, pacient s nejvyšší TF).

In [ ]:
tf = np.array([72, 90, 65, 88])

idx_max = tf.argmax()
idx_min = tf.argmin()

print(f"Nejvyšší TF má pacient {idx_max} (hodnota {tf[idx_max]})")
print(f"Nejnižší TF má pacient {idx_min} (hodnota {tf[idx_min]})")


In [ ]:
# argmax/argmin lze použít i s axis – index extrému v každém řádku/sloupci matice
print("Index maxima v každém sloupci:", mereni.argmax(axis=0))
print("Index maxima v každém řádku: ", mereni.argmax(axis=1))


## 2.6 Ukládání a načítání polí

NumPy má dva vlastní binární formáty, které zachovají dtype a tvar přesně:

| Formát | Funkce | Obsah |
|--------|--------|-------|
| `.npy` | `np.save` / `np.load` | jedno pole |
| `.npz` | `np.savez` / `np.load` | více pojmenovaných polí v jednom souboru |

```python
# .npy – jedno pole
np.save("data.npy", pole)        # uložit
pole = np.load("data.npy")       # načíst → přímo ndarray

# .npz – více polí najednou
np.savez("data.npz", x=cas, y=signal, meta=pole3)  # uložit
d = np.load("data.npz")                             # načíst → dict-like objekt
cas, signal = d["x"], d["y"]                        # přístup přes klíče
```

> **Tip:** `.npz` se hodí, když potřebuješ předat více polí dohromady – v dalším cíli takto načteš
> všechna data pro Matplotlib (EKG, MRI, glykémie…) z jednoho souboru `mpl_data.npz`.

---

### ÚKOL: Analýza dat klinické studie

Stáhni si datový soubor se záznamy 20 pacientů:  
📥 **[pacienti.npy](../pacienti.npy)**

Soubor obsahuje 2D pole tvaru `(20, 3)` – každý řádek je jeden pacient, sloupce jsou:
| Sloupec | Ukazatel | Jednotka |
|---------|----------|----------|
| 0 | Systolický krevní tlak | mmHg |
| 1 | Diastolický krevní tlak | mmHg |
| 2 | Tepová frekvence | BPM |

Formát `.npy` je binární formát NumPy – rychlý a přesný (zachovává dtype, tvar). Načteš ho jednoduše:
```python
data = np.load("pacienti.npy")
```

**1.** Načti soubor `pacienti.npy` a vypiš tvar pole a prvních 5 řádků.

**2.** Vypiš průměr a směrodatnou odchylku každého ukazatele.

**3.** Pomocí boolean indexingu najdi pacienty s hypertenzí – systolický tlak ≥ 140 mmHg. Vypiš jejich počet a průměrný systolický tlak.

**4.** Přidej 4. sloupec: **pulzní tlak** = systolický − diastolický. Použij `np.column_stack()` nebo `np.hstack()`.

**5.** Zjisti, který pacient má nejvyšší tepovou frekvenci – vypiš jeho index a celý záznam. Použij `np.argmax()`.


In [ ]:
import numpy as np

# 1. Načtení dat
data = np.load("pacienti.npy")


# 2. Průměr a std každého ukazatele


# 3. Pacienti s hypertenzí (systolický ≥ 140)


# 4. Pulzní tlak jako nový sloupec


# 5. Pacient s nejvyšší tepovou frekvencí

